# Retina from scratch: Stage 1 training run on a Kaggle GPU

This notebook runs the repo's scripts on a free Kaggle GPU:
1. check the GPU, 2. get the code from GitHub, 3. point at the data, 4. preprocess the photos,
5. a quick sanity check, 6. the full training run (from random weights, as always), 7. results.

**One-time setup**, in the right-hand panel under *Settings*:
- **Accelerator: GPU T4 x2.** Not P100: current PyTorch builds dropped support for the P100's GPU
  generation, so it fails with `no kernel image is available`. Training uses one of the two T4s.
- **Internet: on**, so step 2 can `git clone` the code (Kaggle asks for a phone-verified account for this).
- **Add Input:** a copy of the EyePACS 2015 training photos together with `trainLabels.csv`. The photos
  must be unpacked image files (`.jpeg`), not the original competition's split `train.zip.001`... archives.
  Follow the dataset's license terms.

**How to run:** run steps 1-5 one by one in the editor to check that everything is wired up.
Then use **Save Version → Save & Run All** for the full run: it keeps going after you close the browser,
and everything written to `/kaggle/working` (code version, checkpoint, history, test results)
is saved as that version's output.

## 1. Check the GPU

In [ ]:
import torch

print("PyTorch", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Settings -> Accelerator -> GPU T4 x2.")
print("GPU:", torch.cuda.get_device_name(0))
try:
    torch.ones(1, device="cuda").add_(1)   # a tiny real GPU operation...
    torch.cuda.synchronize()               # ...and wait for it, so a problem shows up here
except RuntimeError as e:
    raise RuntimeError("PyTorch can't run on this GPU. On a P100 that is expected: current builds "
                       "dropped support for it. Settings -> Accelerator -> GPU T4 x2.") from e
print("GPU works.")

## 2. Get the code

Downloads the newest code from GitHub into `/kaggle/working`, so the exact code version is saved next to
the results. (To rerun an old version exactly, put its commit in `CODE_REF`.) Kaggle already has PyTorch, torchvision, scikit-learn, pandas and Pillow installed.
Don't `pip install torch` here: the preinstalled build is matched to Kaggle's GPU drivers.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/AndrewStrong615/Retna.Ai"
CODE_DIR = "/kaggle/working/Retna.Ai"
CODE_REF = ""   # "" = the newest code; or a commit such as "b5488d4" to rerun an old version exactly
os.environ["PYTHONUNBUFFERED"] = "1"   # make the scripts print each line right away


def run(cmd):
    """Run a shell command, show its output as it arrives, and stop the notebook if it fails.
    (A plain `!command` line would NOT stop "Run All" when the command fails.)"""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:   # the notebook's Stop button stops the command too
        proc.kill()
        raise
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {proc.returncode}): {cmd}")


if not os.path.exists(CODE_DIR):
    run(f"git clone --quiet {REPO_URL} {CODE_DIR}")
run(f"git -C {CODE_DIR} fetch --quiet origin")     # works the same on a first run and a re-run
run(f"git -C {CODE_DIR} checkout --quiet --detach {CODE_REF or 'origin/HEAD'}")
os.chdir(CODE_DIR)
run("git log -1 --format='Code version: %h %s'")

## 3. Point to the data

The settings cell already points at your dataset (berinaniesh/diabetic-retinopathy-detection) and turns
enhance on. **For each new run, change only `EXPERIMENT`** (and `TRAIN_ARGS` if the experiment needs an option).
The listing cell shows what Kaggle mounted, in case you ever attach a different dataset; then `RAW_DIR` must be
the folder that directly contains the photos. The check cell makes sure labels and photos match up.

In [ ]:
import glob
import os

# What did Kaggle mount? Label files, and folders holding at least 100 images.
for path in sorted(glob.glob("/kaggle/input/**/*.csv*", recursive=True)):
    print("CSV    ", path)
for folder, _, files in sorted(os.walk("/kaggle/input")):
    n_images = sum(f.lower().endswith((".jpeg", ".jpg", ".png", ".tif", ".tiff")) for f in files)
    if n_images >= 100:
        print(f"IMAGES  {folder}  ({n_images} files)")

In [ ]:
# ---- The data: your Kaggle dataset (change only if you attach a different one) ----
DATA = "/kaggle/input/datasets/berinaniesh/diabetic-retinopathy-detection/data"
LABELS_CSV = f"{DATA}/trainLabels.csv"   # columns: image,level
RAW_DIR = f"{DATA}/train"                # folder that directly holds the photos

# ---- This experiment ----
EXPERIMENT = "exp3"   # a NEW name for every run, so results never overwrite each other
TRAIN_ARGS = ""       # extra train.py options, e.g. "--target binary" (exp4) or "--epochs 60"
IMAGE_SIZE = 384      # the default in preprocess.py and train.py
ENHANCE = True        # local average color subtraction: exp2 showed it helps a lot

RUN_DIR = f"/kaggle/working/runs/{EXPERIMENT}"   # checkpoint, history, test results (saved as output)

# Preprocessed photos go to scratch disk, not /kaggle/working (see step 4).
PROCESSED_DIR = f"/tmp/processed_{IMAGE_SIZE}" + ("_enhanced" if ENHANCE else "")
print(f"Settings saved: {EXPERIMENT} | {TRAIN_ARGS or 'default options'} | photos: {RAW_DIR}")

In [ ]:
import os
import pandas as pd

labels = pd.read_csv(LABELS_CSV)
if not {"image", "level"} <= set(labels.columns):
    raise RuntimeError(f"Expected columns image,level in {LABELS_CSV}, found {list(labels.columns)}")
stems = {os.path.splitext(f)[0] for f in os.listdir(RAW_DIR)}
matched = labels["image"].astype(str).isin(stems).mean()
print(f"{len(labels)} labels | {len(stems)} files in RAW_DIR | {matched:.1%} of labels have a photo")
print("Labels per grade (0 = no DR ... 4 = proliferative):",
      labels["level"].value_counts().sort_index().to_dict())
if matched < 0.5:
    raise RuntimeError("Most labels have no matching photo. Check LABELS_CSV and RAW_DIR.")

## 4. Preprocess the photos (once per session)

Crops, pads and resizes every photo (see `preprocess.py`): roughly 5-20 minutes, depending on how large
the input photos are. The output goes to scratch disk because tens of thousands of files would run into
Kaggle's limit on output files, so each new session (including Save & Run All) redoes this step.
Photos that fail to open are listed and skipped; `train.py` then skips their labels too.

In [ ]:
done_marker = os.path.join(PROCESSED_DIR, ".done")
if os.path.exists(done_marker):
    print("Already preprocessed in this session:", PROCESSED_DIR)
else:
    run(f"python preprocess.py --src {RAW_DIR} --dst {PROCESSED_DIR} --size {IMAGE_SIZE}"
        + (" --enhance" if ENHANCE else ""))
    open(done_marker, "w").close()

## 5. Sanity check (a few minutes)

2 epochs on 2,000 photos. This is the first time the GPU-only code path runs (mixed precision with
`GradScaler`), which the CPU smoke test couldn't cover, so problems show up here in minutes instead of
hours. Ignore the AUC: the model has barely trained. It writes to `/tmp`, so nothing here is saved.

In [ ]:
run(f"python train.py --labels {LABELS_CSV} --images {PROCESSED_DIR} --size {IMAGE_SIZE} "
    f"--limit 2000 --epochs 2 {TRAIN_ARGS} --out /tmp/runs/sanity --overwrite")

## 6. Full training run

Uses `train.py`'s defaults plus `TRAIN_ARGS`: learning from the full 0-4 grade (`--target ordinal`), 30 epochs,
batch 32, AdamW (learning rate 1e-3, weight decay 0.05), and a decision threshold picked on validation for 90%
sensitivity. Expect roughly 1.5-2 hours on a T4: epochs are a little slower than exp2's 2.4 min because the
rotation augmentation is now smoother (bilinear). After epoch 1 it prints its own estimate. The test set is
evaluated once, at the very end. `train.py` refuses to write into a folder that already holds results,
so give every run a new `EXPERIMENT` name.

In [ ]:
run(f"python train.py --labels {LABELS_CSV} --images {PROCESSED_DIR} --size {IMAGE_SIZE} "
    f"{TRAIN_ARGS} --out {RUN_DIR}")

## 7. Results

In [ ]:
import json

with open(f"{RUN_DIR}/history.json") as f:
    history = pd.DataFrame(json.load(f))
columns = ["epoch", "train_loss", "val_loss", "train_auc", "val_auc", "val_specificity", "minutes", "data_wait"]
display(history[[c for c in columns if c in history]].round(4))
results_path = f"{RUN_DIR}/test_results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        print("Test set, evaluated once:", json.load(f))
else:
    print("No test results: this run saved no checkpoint (see the log above).")

### Reading the numbers
- **val_auc** measures how well the model ranks sick eyes above healthy ones (0.5 = coin flip, 1.0 = perfect).
  The epoch with the best val_auc is the one saved in `best.pt`.
- **Test sensitivity and specificity** use the threshold picked on validation for 90% sensitivity,
  so test sensitivity lands near 90%, not exactly on it.
- **Compare experiments by validation AUC, never by test results.** Picking between runs by their
  test score quietly turns the test set into a second validation set.